# Lab 01 — Part D: Experiment 1 — Inspect the Sparse Representation

Notebook thực hiện đầy đủ các yêu cầu từ mục 7.1 đến 7.5 của đề: đọc corpus, tạo pipeline Count → TF → IDF → TF-IDF, kiểm tra kích thước/sparsity và inspect vocabulary.

In [5]:
# 7.1 — Dataset: tìm và đọc toàn bộ corpus C4 30K documents.
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

DATASET_PATH_PARTS = (
    "data",
    "c4-train.00000-of-01024-30K.json",
    "c4-train.00000-of-01024-30K.json",
)
TEXT_FIELD = "text"
WORKING_DIRECTORY = Path.cwd().resolve()
CANDIDATE_DATASET_PATHS = tuple(
    directory.joinpath(*DATASET_PATH_PARTS)
    for directory in (WORKING_DIRECTORY, *WORKING_DIRECTORY.parents)
)
DATASET_PATH = next(
    path for path in CANDIDATE_DATASET_PATHS if path.is_file()
)

with DATASET_PATH.open(encoding="utf-8") as dataset_file:
    documents = tuple(
        record[TEXT_FIELD].strip()
        for line in dataset_file
        for record in (json.loads(line),)
        if isinstance(record.get(TEXT_FIELD), str) and record[TEXT_FIELD].strip()
    )

print(f"Dataset: {DATASET_PATH}")
print(f"Number of non-empty documents: {len(documents):,}")


Dataset: G:\My Drive\Documents\NLP\data\c4-train.00000-of-01024-30K.json\c4-train.00000-of-01024-30K.json
Number of non-empty documents: 30,000


In [6]:
# 7.2 — Pipeline: Raw text → tokenizer/CountVectorizer → TF → IDF → TF-IDF.
TOKEN_PATTERN = r"(?u)\b[^\W_]+(?:['-][^\W_]+)*\b"
COUNT_DTYPE = np.int64

count_vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=TOKEN_PATTERN,
    dtype=COUNT_DTYPE,
)
count_matrix = count_vectorizer.fit_transform(documents).tocsr()
vocabulary = count_vectorizer.get_feature_names_out()
document_count, vocabulary_size = count_matrix.shape
document_frequencies = np.asarray(count_matrix.sign().sum(axis=0)).ravel()
idf_values = np.log(document_count / document_frequencies)
document_token_totals = np.asarray(count_matrix.sum(axis=1)).ravel()
term_frequency_matrix = count_matrix.multiply(
    1.0 / document_token_totals[:, np.newaxis]
).tocsr()
tf_idf_matrix = term_frequency_matrix.multiply(idf_values).tocsr()

print("Pipeline completed: Count → TF → IDF → TF-IDF")


Pipeline completed: Count → TF → IDF → TF-IDF


In [7]:
# 7.3 — Kích thước: number of documents, vocabulary size và matrix shape.
matrix_shape = tf_idf_matrix.shape

print(f"Number of documents (N): {document_count:,}")
print(f"Vocabulary size (V): {vocabulary_size:,}")
print(f"TF-IDF matrix shape: {matrix_shape}")


Number of documents (N): 30,000
Vocabulary size (V): 233,866
TF-IDF matrix shape: (30000, 233866)


In [8]:
# 7.4 — Sparsity: S = 1 - nnz(X) / (N × V).
matrix_entry_count = document_count * vocabulary_size
nonzero_entry_count = tf_idf_matrix.nnz
sparsity = 1.0 - (nonzero_entry_count / matrix_entry_count)

print(f"Non-zero TF-IDF entries: {nonzero_entry_count:,}")
print(f"Total matrix entries: {matrix_entry_count:,}")
print(f"Sparsity: {sparsity:.6%}")


Non-zero TF-IDF entries: 5,080,798
Total matrix entries: 7,015,980,000
Sparsity: 99.927582%


In [9]:
# 7.5a — Inspect vocabulary: 20 terms phổ biến nhất theo document frequency.
TOP_TERM_COUNT = 20
STABLE_SORT_KIND = "stable"
frequency_order = np.argsort(-document_frequencies, kind=STABLE_SORT_KIND)[:TOP_TERM_COUNT]
frequent_term_rows = tuple(
    f"{rank:>2}. {vocabulary[index]:<24} df={int(document_frequencies[index]):,}"
    for rank, index in enumerate(frequency_order, start=1)
)

print("20 terms có document frequency cao nhất")
print("\n".join(frequent_term_rows))


20 terms có document frequency cao nhất
 1. the                      df=27,890
 2. and                      df=27,418
 3. to                       df=26,665
 4. of                       df=26,013
 5. a                        df=25,880
 6. in                       df=25,129
 7. for                      df=23,643
 8. is                       df=22,738
 9. with                     df=21,403
10. on                       df=20,095
11. that                     df=18,274
12. this                     df=17,837
13. are                      df=17,594
14. it                       df=16,762
15. as                       df=16,454
16. at                       df=16,326
17. from                     df=16,310
18. be                       df=16,146
19. you                      df=15,933
20. by                       df=15,072


In [10]:
# 7.5b — Inspect vocabulary: 20 terms có IDF cao nhất.
idf_order = np.argsort(-idf_values, kind=STABLE_SORT_KIND)[:TOP_TERM_COUNT]
high_idf_rows = tuple(
    f"{rank:>2}. {vocabulary[index]:<24} idf={idf_values[index]:.6f}"
    for rank, index in enumerate(idf_order, start=1)
)

print("20 terms có IDF cao nhất")
print("\n".join(high_idf_rows))


20 terms có IDF cao nhất
 1. 0's                      idf=10.308953
 2. 0-0-1-2-3-4-5-6          idf=10.308953
 3. 0-06-464075-2            idf=10.308953
 4. 0-1-0                    idf=10.308953
 5. 0-100km                  idf=10.308953
 6. 0-100mph                 idf=10.308953
 7. 0-105                    idf=10.308953
 8. 0-10v                    idf=10.308953
 9. 0-11                     idf=10.308953
10. 0-17                     idf=10.308953
11. 0-200                    idf=10.308953
12. 0-255                    idf=10.308953
13. 0-262-69339-9            idf=10.308953
14. 0-27                     idf=10.308953
15. 0-3-0                    idf=10.308953
16. 0-306-81326-2            idf=10.308953
17. 0-316-47282-4            idf=10.308953
18. 0-38                     idf=10.308953
19. 0-3cm                    idf=10.308953
20. 0-4                      idf=10.308953


In [11]:
# 7.5c — Inspect vocabulary: 20 terms TF-IDF cao nhất trong một document được chọn.
SELECTED_DOCUMENT_INDEX = 0
selected_document_vector = tf_idf_matrix.getrow(SELECTED_DOCUMENT_INDEX)
selected_term_order = np.argsort(
    -selected_document_vector.data,
    kind=STABLE_SORT_KIND,
)[:TOP_TERM_COUNT]
selected_term_indices = selected_document_vector.indices[selected_term_order]
selected_term_weights = selected_document_vector.data[selected_term_order]
selected_term_rows = tuple(
    f"{rank:>2}. {vocabulary[index]:<24} tf-idf={weight:.6f}"
    for rank, (index, weight) in enumerate(
        zip(selected_term_indices, selected_term_weights),
        start=1,
    )
)

print(f"Selected document index: {SELECTED_DOCUMENT_INDEX}")
print("20 terms có TF-IDF cao nhất trong document đã chọn")
print("\n".join(selected_term_rows))


Selected document index: 0
20 terms có TF-IDF cao nhất trong document đã chọn
 1. bbq                      tf-idf=0.180142
 2. class                    tf-idf=0.099743
 3. balay                    tf-idf=0.079300
 4. kcbs                     tf-idf=0.073968
 5. meat                     tf-idf=0.073531
 6. lonestar                 tf-idf=0.070849
 7. missoula                 tf-idf=0.061587
 8. smoker                   tf-idf=0.058468
 9. apron                    tf-idf=0.057066
10. timelines                tf-idf=0.052884
11. trimming                 tf-idf=0.051951
12. spectators               tf-idf=0.050924
13. rangers                  tf-idf=0.049363
14. 22nd                     tf-idf=0.046842
15. beginner                 tf-idf=0.046088
16. t-shirt                  tf-idf=0.045592
17. beginners                tf-idf=0.045402
18. culinary                 tf-idf=0.045216
19. cost                     tf-idf=0.044208
20. tasting                  tf-idf=0.043723


## Câu hỏi phân tích cần tự trả lời

1. Vì sao mỗi document chỉ dùng một phần nhỏ vocabulary nhưng vector vẫn có V chiều?
2. Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?
3. Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?

Dựa vào ba danh sách ở mục 7.5 để trả lời bằng evidence từ lần chạy của bạn.

## Trả lời câu hỏi phân tích

### 1. Vì sao document dùng ít term nhưng vector vẫn có V chiều?

Vocabulary toàn corpus xác định một feature space chung có V = 233.866 chiều. Mỗi document được đặt vào cùng không gian đó để có thể so sánh với mọi document khác. Document chỉ có trọng số khác 0 ở các term mà nó chứa; các chiều còn lại bằng 0. Vì thế ma trận đạt sparsity 99,927582%, nhưng mỗi hàng vẫn có đúng 233.866 chiều.

### 2. Term xuất hiện nhiều có nhất thiết có TF-IDF cao không?

Không. Theo công thức IDF, document frequency càng lớn thì IDF càng nhỏ. Trong kết quả, the xuất hiện ở 27.890/30.000 document, and ở 27.418 document và to ở 26.665 document; các term này có trọng số phân biệt thấp. Một term phổ biến chỉ có TF-IDF đáng kể trong một document nếu TF của nó tại document đó đủ lớn, nhưng nó không tự động đứng đầu danh sách TF-IDF.

### 3. Term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?

Không. TF-IDF là tích của TF và IDF, nên term không xuất hiện trong document có TF = 0 và TF-IDF = 0, dù IDF toàn corpus rất cao. Danh sách IDF cao nhất chứa các term hiếm như 0-100km và 0-100mph, trong khi top TF-IDF của document 0 là bbq, class và balay. Điều này cho thấy IDF cao phải đi kèm việc term xuất hiện trong document và có TF phù hợp mới tạo ra trọng số TF-IDF cao.